# Multi-agent concept-policy workflow

This notebook demonstrates concept supervision and intervention inspired by [Zabounidis et al. (2023)](https://proceedings.mlr.press/v205/zabounidis23a.html).

It uses synthetic categorical observations and supervised expert-action labels, not FortAttack rollouts or MAPPO. The metrics describe this classifier example only.


## Experiment


In [1]:
import copy
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import SteeringVectors
from tdhook.workflow import Workflow
from xdrl import interpret

SEED = 5900
TRAIN_EPISODES = 256
EVALUATION_EPISODES = 96
AGENTS = 2
OBSERVATION_DIM = 12
HIDDEN_DIM = 16
CONCEPT_DIM = 7
ACTION_ORDER = ("TURN_LEFT", "TURN_RIGHT", "ADVANCE", "WAIT", "TAG")
CONCEPT_GROUPS = {"strategy": slice(0, 3), "target": slice(3, 5), "range": slice(5, 7)}
TRAINING_SEEDS = (5900, 5901, 5902, 5903, 5904)
TRAINING_STEPS = 80
LEARNING_RATE = 0.03
CONCEPT_LOSS_COEFFICIENT = 1.0
torch.manual_seed(SEED)

## Generate labeled examples


In [2]:
def make_fixture(episodes, *, seed, noise):
    generator = torch.Generator().manual_seed(seed)
    strategy = torch.randint(3, (episodes, 1), generator=generator).expand(-1, AGENTS).clone()
    target = torch.randint(2, (episodes, AGENTS), generator=generator)
    in_range = torch.randint(2, (episodes, AGENTS), generator=generator)
    strategy_one_hot = torch.nn.functional.one_hot(strategy, 3).float()
    target_one_hot = torch.nn.functional.one_hot(target, 2).float()
    range_one_hot = torch.nn.functional.one_hot(in_range, 2).float()
    concepts = torch.cat((strategy_one_hot, target_one_hot, range_one_hot), dim=-1)
    mixing = torch.tensor(
        [
            [1.0, -0.4, 0.2, 0.7, -0.2, 0.4, -0.6],
            [-0.3, 0.9, 0.1, -0.6, 0.8, -0.5, 0.3],
            [0.2, -0.5, 1.0, 0.4, 0.1, 0.7, -0.3],
            [0.8, 0.2, -0.4, 0.5, -0.7, 0.2, 0.6],
            [-0.6, 0.1, 0.7, -0.2, 0.6, 0.8, -0.4],
            [0.4, 0.6, -0.3, 0.9, 0.2, -0.6, 0.1],
            [0.1, -0.7, 0.5, 0.3, 0.9, -0.1, 0.4],
            [0.5, 0.3, 0.6, -0.8, 0.1, 0.2, 0.7],
        ]
    )
    signal = concepts @ mixing.T
    nuisance = torch.randn(episodes, AGENTS, 4, generator=generator)
    observation = torch.cat((signal + noise * torch.randn(signal.shape, generator=generator), nuisance), dim=-1)
    action = torch.where(in_range.bool(), torch.full_like(strategy, 4), (strategy + 2 * target) % 4)
    return {
        "episode_id": torch.arange(seed * 10000, seed * 10000 + episodes),
        "observation": observation,
        "concept": concepts,
        "strategy": strategy,
        "target": target,
        "range": in_range,
        "action": action,
    }


train_fixture = make_fixture(TRAIN_EPISODES, seed=SEED + 10, noise=0.28)
evaluation_fixture = make_fixture(EVALUATION_EPISODES, seed=SEED + 20, noise=0.62)
assert set(train_fixture["episode_id"].tolist()).isdisjoint(evaluation_fixture["episode_id"].tolist())
assert evaluation_fixture["observation"].shape == (EVALUATION_EPISODES, AGENTS, OBSERVATION_DIM)
{
    "task": "compact-2v2-defense",
    "ordered_batch_axes": ("episode", "agent"),
    "concept_groups": {name: (group.start, group.stop) for name, group in CONCEPT_GROUPS.items()},
    "action_order": ACTION_ORDER,
}

{'task': 'compact-2v2-defense',
 'ordered_batch_axes': ('episode', 'agent'),
 'concept_groups': {'strategy': (0, 3), 'target': (3, 5), 'range': (5, 7)},
 'action_order': ('TURN_LEFT', 'TURN_RIGHT', 'ADVANCE', 'WAIT', 'TAG')}

## Train matched classifiers


In [3]:
class BottleneckPolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = torch.nn.Sequential(torch.nn.Linear(OBSERVATION_DIM, HIDDEN_DIM), torch.nn.Tanh())
        self.concept_head = torch.nn.Linear(HIDDEN_DIM, CONCEPT_DIM)
        self.action_head = torch.nn.Linear(CONCEPT_DIM, len(ACTION_ORDER))

    def forward(self, observation):
        concept_logits = self.concept_head(self.encoder(observation))
        concept_probabilities = torch.cat(
            [torch.softmax(concept_logits[..., group], dim=-1) for group in CONCEPT_GROUPS.values()], dim=-1
        )
        return (concept_logits, self.action_head(concept_probabilities))


def grouped_concept_loss(logits, fixture):
    losses = []
    for name, group in CONCEPT_GROUPS.items():
        losses.append(torch.nn.functional.cross_entropy(logits[..., group].flatten(0, 1), fixture[name].flatten()))
    return torch.stack(losses).mean()


def action_accuracy(logits, target):
    return float((logits.argmax(-1) == target).float().mean())


def train_pair(seed):
    torch.manual_seed(seed)
    initial = BottleneckPolicy()
    concept_model = copy.deepcopy(initial)
    baseline_model = copy.deepcopy(initial)
    concept_optimizer = torch.optim.Adam(concept_model.parameters(), lr=LEARNING_RATE)
    baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)
    curves = {"concept": [], "baseline": []}
    for _step in range(TRAINING_STEPS):
        for name, model, optimizer, auxiliary in (
            ("concept", concept_model, concept_optimizer, True),
            ("baseline", baseline_model, baseline_optimizer, False),
        ):
            optimizer.zero_grad()
            concept_logits, action_logits = model(train_fixture["observation"])
            loss = torch.nn.functional.cross_entropy(action_logits.flatten(0, 1), train_fixture["action"].flatten())
            if auxiliary:
                loss = loss + CONCEPT_LOSS_COEFFICIENT * grouped_concept_loss(concept_logits, train_fixture)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                eval_action_logits = model(evaluation_fixture["observation"])[1]
                curves[name].append(action_accuracy(eval_action_logits, evaluation_fixture["action"]))
    return (concept_model.eval(), baseline_model.eval(), curves)


runs = {seed: train_pair(seed) for seed in TRAINING_SEEDS}
selected_model, selected_baseline, _ = runs[SEED]
parameter_counts = {
    "concept": sum((parameter.numel() for parameter in selected_model.parameters())),
    "baseline": sum((parameter.numel() for parameter in selected_baseline.parameters())),
}
assert parameter_counts["concept"] == parameter_counts["baseline"]
training_budget = {
    "parameter_counts": parameter_counts,
    "examples_per_step": TRAIN_EPISODES * AGENTS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "steps": TRAINING_STEPS,
    "seeds": TRAINING_SEEDS,
    "matched": True,
    "declared_difference": "grouped auxiliary concept loss only",
}
training_budget

{'parameter_counts': {'concept': 367, 'baseline': 367},
 'examples_per_step': 512,
 'optimizer': 'Adam',
 'learning_rate': 0.03,
 'steps': 80,
 'seeds': (5900, 5901, 5902, 5903, 5904),
 'matched': True,
 'declared_difference': 'grouped auxiliary concept loss only'}

## Evaluate held-out examples


In [4]:
def evaluate_model(model):
    with torch.inference_mode():
        concepts, actions = model(evaluation_fixture["observation"])
    group_accuracy = {
        name: float((concepts[..., group].argmax(-1) == evaluation_fixture[name]).float().mean(dim=(0, 1)))
        for name, group in CONCEPT_GROUPS.items()
    }
    correct_by_agent = actions.argmax(-1) == evaluation_fixture["action"]
    return {
        "concept_accuracy_reduction": "mean over (episode, agent), reported per concept_group",
        "concept_accuracy": group_accuracy,
        "policy_accuracy_reduction": "mean over (episode, agent)",
        "agent_action_accuracy": float(correct_by_agent.float().mean(dim=(0, 1))),
        "per_agent_action_accuracy": correct_by_agent.float().mean(dim=0).tolist(),
        "episode_all_agents_correct": float(correct_by_agent.all(dim=1).float().mean(dim=0)),
    }


per_seed = {}
for seed, (concept_model, baseline_model, curves) in runs.items():
    per_seed[seed] = {
        "concept": evaluate_model(concept_model),
        "baseline": evaluate_model(baseline_model),
        "training_curve": {
            arm: {
                "area_mean": float(torch.tensor(values).mean()),
                "first_step_at_0.80": next((index + 1 for index, value in enumerate(values) if value >= 0.8), None),
                "final": values[-1],
            }
            for arm, values in curves.items()
        },
    }


def seed_summary(arm, metric):
    values = torch.tensor([per_seed[seed][arm][metric] for seed in TRAINING_SEEDS])
    return {"mean": float(values.mean()), "sample_std": float(values.std()), "values": values.tolist()}


held_out_summary = {
    "selected_seed": SEED,
    "selected_concept_policy": per_seed[SEED]["concept"],
    "selected_nonconcept_policy": per_seed[SEED]["baseline"],
    "stability_across_training_seeds": {
        arm: {
            "agent_action_accuracy": seed_summary(arm, "agent_action_accuracy"),
            "episode_all_agents_correct": seed_summary(arm, "episode_all_agents_correct"),
        }
        for arm in ("concept", "baseline")
    },
    "descriptive_training_curve_summaries": {seed: per_seed[seed]["training_curve"] for seed in TRAINING_SEEDS},
}
held_out_summary

{'selected_seed': 5900,
 'selected_concept_policy': {'concept_accuracy_reduction': 'mean over (episode, agent), reported per concept_group',
  'concept_accuracy': {'strategy': 0.8072916865348816,
   'target': 0.875,
   'range': 0.9479166865348816},
  'policy_accuracy_reduction': 'mean over (episode, agent)',
  'agent_action_accuracy': 0.8385416865348816,
  'per_agent_action_accuracy': [0.8229166865348816, 0.8541666865348816],
  'episode_all_agents_correct': 0.7083333134651184},
 'selected_nonconcept_policy': {'concept_accuracy_reduction': 'mean over (episode, agent), reported per concept_group',
  'concept_accuracy': {'strategy': 0.53125,
   'target': 0.5416666865348816,
   'range': 0.8541666865348816},
  'policy_accuracy_reduction': 'mean over (episode, agent)',
  'agent_action_accuracy': 0.6979166865348816,
  'per_agent_action_accuracy': [0.65625, 0.7395833134651184],
  'episode_all_agents_correct': 0.4895833432674408},
 'stability_across_training_seeds': {'concept': {'agent_action_a

## Wrap the classifier


In [5]:
policy_core = copy.deepcopy(selected_model).eval()
policy = TensorDictModule(policy_core, in_keys=["observation"], out_keys=["concept_logits", "action_logits"])
evaluation_batch = TensorDict(
    {"observation": evaluation_fixture["observation"].clone()},
    batch_size=[EVALUATION_EPISODES, AGENTS],
    names=["episode", "agent"],
)
component = interpret(policy)
with torch.inference_mode():
    native_concepts, native_actions = policy_core(evaluation_fixture["observation"])
    adapted = component(evaluation_batch.clone())
adapter_parity = torch.equal(native_concepts, adapted["concept_logits"]) and torch.equal(
    native_actions, adapted["action_logits"]
)
assert adapter_parity
{
    "batch_axes": evaluation_batch.names,
    "concept_tensor_shape": tuple(adapted["concept_logits"].shape),
    "action_tensor_shape": tuple(adapted["action_logits"].shape),
    "native_xdrl_output_parity": adapter_parity,
}

{'batch_axes': ['episode', 'agent'],
 'concept_tensor_shape': (96, 2, 7),
 'action_tensor_shape': (96, 2, 5),
 'native_xdrl_output_parity': True}

## Replace concepts


In [6]:
def logits_from_fixture(fixture, *, wrong=False):
    parts = []
    for name, width in (("strategy", 3), ("target", 2), ("range", 2)):
        labels = fixture[name]
        if wrong:
            labels = (labels + 1) % width
        parts.append(16.0 * torch.nn.functional.one_hot(labels, width).float() - 8.0)
    return torch.cat(parts, dim=-1)


oracle_logits = logits_from_fixture(evaluation_fixture)
incorrect_logits = logits_from_fixture(evaluation_fixture, wrong=True)
shuffle_generator = torch.Generator().manual_seed(SEED + 30)
shuffled_logits = oracle_logits[torch.randperm(EVALUATION_EPISODES, generator=shuffle_generator)]


def keep_concepts(*, output, **_):
    return output


def replacement_callback(label, replacement):
    def replace(*, output, **_):
        assert replacement.shape == output.shape
        return replacement.to(device=output.device, dtype=output.dtype)

    replace.__name__ = f"replace_{label}_concepts"
    return replace


callbacks = {
    "correct": replacement_callback("correct", oracle_logits),
    "incorrect": replacement_callback("incorrect", incorrect_logits),
    "shuffled": replacement_callback("shuffled", shuffled_logits),
}


def intervention_context(training_seed):
    core = copy.deepcopy(runs[training_seed][0]).eval()
    seed_policy = TensorDictModule(core, in_keys=["observation"], out_keys=["concept_logits", "action_logits"])
    return {"component": interpret(seed_policy)}


seed_contexts = {training_seed: intervention_context(training_seed) for training_seed in TRAINING_SEEDS}


def run_intervention_pair(training_seed, callback):
    component = seed_contexts[training_seed]["component"]
    torch.manual_seed(training_seed)
    baseline = component.run(
        Workflow(SteeringVectors(["module.concept_head"], steer_fn=keep_concepts)), evaluation_batch.clone()
    )
    torch.manual_seed(training_seed)
    intervention = component.run(
        Workflow(SteeringVectors(["module.concept_head"], steer_fn=callback)), evaluation_batch.clone()
    )
    return {"baseline": baseline, "intervention": intervention}


pairs_by_seed = {
    training_seed: {label: run_intervention_pair(training_seed, callback) for label, callback in callbacks.items()}
    for training_seed in TRAINING_SEEDS
}
{"workflow_target": "module.concept_head", "training_seeds": TRAINING_SEEDS}

{'workflow_target': 'module.concept_head',
 'training_seeds': (5900, 5901, 5902, 5903, 5904)}

## Measure intervention effects


In [7]:
def paired_metrics(action_logits):
    correct = action_logits.argmax(-1) == evaluation_fixture["action"]
    return {
        "agent_correct_by_episode": correct.float().mean(dim=1),
        "all_agents_correct_by_episode": correct.all(dim=1).float(),
        "action_by_episode_agent": action_logits.argmax(-1),
    }


def bootstrap_mean_interval(values, *, seed, draws):
    generator = torch.Generator().manual_seed(seed)
    estimates = []
    for _ in range(draws):
        indices = torch.randint(len(values), (len(values),), generator=generator)
        estimates.append(values[indices].mean())
    return [float(value) for value in torch.stack(estimates).quantile(torch.tensor((0.025, 0.975)))]


def summarize_seed_pair(training_seed, label, pair):
    baseline = paired_metrics(pair["baseline"].data["action_logits"])
    changed = paired_metrics(pair["intervention"].data["action_logits"])
    agent_delta = changed["agent_correct_by_episode"] - baseline["agent_correct_by_episode"]
    team_delta = changed["all_agents_correct_by_episode"] - baseline["all_agents_correct_by_episode"]
    bootstrap_offset = {"correct": 0, "incorrect": 1, "shuffled": 2}[label]
    return {
        "training_seed": training_seed,
        "n_evaluation_episodes": EVALUATION_EPISODES,
        "agent_action_accuracy": {
            "baseline": float(baseline["agent_correct_by_episode"].mean()),
            "intervention": float(changed["agent_correct_by_episode"].mean()),
            "paired_delta": float(agent_delta.mean()),
            "within_seed_episode_bootstrap_95pct": bootstrap_mean_interval(
                agent_delta, seed=training_seed + 100 + bootstrap_offset, draws=500
            ),
        },
        "episode_all_agents_correct": {
            "baseline": float(baseline["all_agents_correct_by_episode"].mean()),
            "intervention": float(changed["all_agents_correct_by_episode"].mean()),
            "paired_delta": float(team_delta.mean()),
            "within_seed_episode_bootstrap_95pct": bootstrap_mean_interval(
                team_delta, seed=training_seed + 200 + bootstrap_offset, draws=500
            ),
        },
        "behavior_change_rate": float(
            (changed["action_by_episode_agent"] != baseline["action_by_episode_agent"]).float().mean(dim=(0, 1))
        ),
    }


per_seed_intervention_results = {
    training_seed: {label: summarize_seed_pair(training_seed, label, pair) for label, pair in seed_pairs.items()}
    for training_seed, seed_pairs in pairs_by_seed.items()
}


def aggregate_metric(label, metric, *, seed):
    baseline = torch.tensor(
        [per_seed_intervention_results[training_seed][label][metric]["baseline"] for training_seed in TRAINING_SEEDS]
    )
    intervention = torch.tensor(
        [
            per_seed_intervention_results[training_seed][label][metric]["intervention"]
            for training_seed in TRAINING_SEEDS
        ]
    )
    paired_delta = intervention - baseline
    return {
        "n_training_seeds": len(TRAINING_SEEDS),
        "training_seed_ids": list(TRAINING_SEEDS),
        "evaluation_episodes_per_seed": EVALUATION_EPISODES,
        "baseline_mean_across_seeds": float(baseline.mean()),
        "intervention_mean_across_seeds": float(intervention.mean()),
        "paired_delta_by_seed": paired_delta.tolist(),
        "mean_paired_delta_across_seeds": float(paired_delta.mean()),
        "training_seed_bootstrap_95pct": bootstrap_mean_interval(paired_delta, seed=seed, draws=5000),
    }


intervention_results = {
    label: {
        "reduction_contract": {
            "within_seed": "paired mean over frozen episodes; agent axis retained within episode",
            "across_seed": "one paired mean per independently trained model; bootstrap over training_seed",
        },
        "agent_action_accuracy": aggregate_metric(label, "agent_action_accuracy", seed=SEED + 300 + index),
        "episode_all_agents_correct": aggregate_metric(label, "episode_all_agents_correct", seed=SEED + 400 + index),
        "per_seed": {
            training_seed: per_seed_intervention_results[training_seed][label] for training_seed in TRAINING_SEEDS
        },
    }
    for index, label in enumerate(callbacks)
}
correct_lower = intervention_results["correct"]["agent_action_accuracy"]["training_seed_bootstrap_95pct"][0]
correct_gain = intervention_results["correct"]["agent_action_accuracy"]["mean_paired_delta_across_seeds"]
largest_control_gain = max(
    (
        intervention_results[label]["agent_action_accuracy"]["mean_paired_delta_across_seeds"]
        for label in ("incorrect", "shuffled")
    )
)
directional_effect_summary = {
    "uncertainty_unit": "training_seed",
    "n_training_seeds": len(TRAINING_SEEDS),
    "correct_seed_bootstrap_lower_bound_nonnegative": correct_lower >= 0.0,
    "correct_mean_gain_exceeds_controls": correct_gain > largest_control_gain,
    "supports_expected_direction": correct_lower >= 0.0 and correct_gain > largest_control_gain,
    "scope": "synthetic smoke fixture only",
}
{"effects": intervention_results, "directional_summary": directional_effect_summary}

{'effects': {'correct': {'reduction_contract': {'within_seed': 'paired mean over frozen episodes; agent axis retained within episode',
    'across_seed': 'one paired mean per independently trained model; bootstrap over training_seed'},
   'agent_action_accuracy': {'n_training_seeds': 5,
    'training_seed_ids': [5900, 5901, 5902, 5903, 5904],
    'evaluation_episodes_per_seed': 96,
    'baseline_mean_across_seeds': 0.8041666746139526,
    'intervention_mean_across_seeds': 0.8979166746139526,
    'paired_delta_by_seed': [0.08854162693023682,
     0.078125,
     0.1302083134651184,
     0.08854162693023682,
     0.08333331346511841],
    'mean_paired_delta_across_seeds': 0.09374997764825821,
    'training_seed_bootstrap_95pct': [0.08229164779186249,
     0.11249997466802597]},
   'episode_all_agents_correct': {'n_training_seeds': 5,
    'training_seed_ids': [5900, 5901, 5902, 5903, 5904],
    'evaluation_episodes_per_seed': 96,
    'baseline_mean_across_seeds': 0.6625000238418579,
    'i

## Results


In [8]:
{
    "held_out": held_out_summary,
    "training": training_budget,
    "interventions": intervention_results,
    "direction": directional_effect_summary,
}

{'held_out': {'selected_seed': 5900,
  'selected_concept_policy': {'concept_accuracy_reduction': 'mean over (episode, agent), reported per concept_group',
   'concept_accuracy': {'strategy': 0.8072916865348816,
    'target': 0.875,
    'range': 0.9479166865348816},
   'policy_accuracy_reduction': 'mean over (episode, agent)',
   'agent_action_accuracy': 0.8385416865348816,
   'per_agent_action_accuracy': [0.8229166865348816, 0.8541666865348816],
   'episode_all_agents_correct': 0.7083333134651184},
  'selected_nonconcept_policy': {'concept_accuracy_reduction': 'mean over (episode, agent), reported per concept_group',
   'concept_accuracy': {'strategy': 0.53125,
    'target': 0.5416666865348816,
    'range': 0.8541666865348816},
   'policy_accuracy_reduction': 'mean over (episode, agent)',
   'agent_action_accuracy': 0.6979166865348816,
   'per_agent_action_accuracy': [0.65625, 0.7395833134651184],
   'episode_all_agents_correct': 0.4895833432674408},
  'stability_across_training_seeds'